# Pré-processamento 3 — Arapiraca

Este notebook transforma os dados de ocorrências em matrizes de séries temporais:
1. Carrega o CSV com índice de célula do grid
2. Calcula a hora do ano para cada ocorrência
3. Para cada tipo de crime, cria uma matriz (célula × hora) por ano
4. Agrega todos os anos e salva um CSV por tipo de crime

**Entrada:** `arapiraca_with_grid_index.csv`

**Saída:** 5 arquivos `arapiraca_{tipo_crime}_all_grid.csv`

## 1. Carregamento dos dados com índice de célula

In [1]:
# Carrega o CSV de Arapiraca com a coluna 'cell' (índice da célula do grid)
# Converte DATA_HORA para datetime

import pandas as pd

df_crimes = pd.read_csv(
    "./output/arapiraca/arapiraca_with_grid_index.csv",
    delimiter=";",
    parse_dates=["DATA_HORA"]
)

## 2. Exploração dos dados

In [2]:
display(df_crimes)
df_crimes.info()
years = sorted(df_crimes['DATA_HORA'].dt.year.unique())
display(years)

,Unnamed: 0,LONGITUDE,LATITUDE,ROUBO_GROUP,DATA_HORA,CIDADE_FATO,cell
0,0,-36.690147,-9.735983,street_robbery,2022-01-01 03:00:00,Arapiraca,2256
1,1,-36.655517,-9.785501,street_robbery,2022-01-01 20:00:00,Arapiraca,1732
2,2,-36.591520,-9.762967,vehicle_robbery,2022-01-02 09:00:00,Arapiraca,768
3,3,-36.664139,-9.768197,street_robbery,2022-01-02 23:00:00,Arapiraca,1850
4,4,-36.680604,-9.644294,street_robbery,2022-01-03 08:00:00,Arapiraca,2163
...,...,...,...,...,...,...,...
17366,17366,-36.662065,-9.764001,vehicle_robbery,2014-12-21 01:30:00,Arapiraca,1851
17367,17367,-36.666260,-9.753766,street_robbery,2014-12-22 14:30:00,Arapiraca,1910
17368,17368,-36.663400,-9.748675,street_robbery,2014-12-22 22:00:00,Arapiraca,1854
17369,17369,-36.648386,-9.721165,vehicle_robbery,2014-12-22 23:30:00,Arapiraca,1633


<class 'pandas.DataFrame'>
RangeIndex: 17371 entries, 0 to 17370
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Unnamed: 0   17371 non-null  int64         
 1   LONGITUDE    17371 non-null  float64       
 2   LATITUDE     17371 non-null  float64       
 3   ROUBO_GROUP  17371 non-null  str           
 4   DATA_HORA    17371 non-null  datetime64[us]
 5   CIDADE_FATO  17371 non-null  str           
 6   cell         17371 non-null  int64         
dtypes: datetime64[us](1), float64(2), int64(2), str(2)
memory usage: 950.1 KB


[np.int32(2012),
 np.int32(2013),
 np.int32(2014),
 np.int32(2015),
 np.int32(2016),
 np.int32(2017),
 np.int32(2018),
 np.int32(2019),
 np.int32(2020),
 np.int32(2021),
 np.int32(2022)]

## 3. Tipos de crime e intervalo temporal

In [3]:
# Identifica os tipos de crime e o intervalo de anos nos dados

crimes_types = sorted(df_crimes['ROUBO_GROUP'].unique())
display(crimes_types)

['commercial_robbery',
 'public_transport_robbery',
 'residential_robbery',
 'street_robbery',
 'vehicle_robbery']

## 4. Cálculo da hora do ano

Para cada ocorrência, calcula quantas horas se passaram desde o início do ano. Isso permite organizar os dados em uma grade temporal uniforme.

In [4]:
# Calcula a hora do ano para cada ocorrência
# Exemplo: 1 de janeiro 00:00 = hora 0, 2 de janeiro 00:00 = hora 24
# Utiliza pandarallel para processamento paralelo

import numpy as np
import datetime
from pandarallel import pandarallel
import os

cpu_count = os.cpu_count() or 1
nb_workers = max(1, min(cpu_count, 24))
pandarallel.initialize(nb_workers=nb_workers, progress_bar=True)


def hour_of_year(dt):
    beginning_of_year = datetime.datetime(
        dt["DATA_HORA"].year, 1, 1, tzinfo=dt["DATA_HORA"].tzinfo
    )
    return pd.Series(
        {
            "hour_year": (dt["DATA_HORA"] - beginning_of_year).total_seconds()
            // 3600
        }
    )


df_crimes
display("Translate date of crime to hour of the year")
df_hour = df_crimes.merge(
    df_crimes.parallel_apply(hour_of_year, axis=1), left_index=True, right_index=True
)
display("Initial shape: ", df_hour.shape)
display("Initial shape: ", df_hour)

INFO: Pandarallel will run on 24 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


'Translate date of crime to hour of the year'

'Initial shape: '

(17371, 8)

'Initial shape: '

,Unnamed: 0,LONGITUDE,LATITUDE,ROUBO_GROUP,DATA_HORA,CIDADE_FATO,cell,hour_year
0,0,-36.690147,-9.735983,street_robbery,2022-01-01 03:00:00,Arapiraca,2256,3.0
1,1,-36.655517,-9.785501,street_robbery,2022-01-01 20:00:00,Arapiraca,1732,20.0
2,2,-36.591520,-9.762967,vehicle_robbery,2022-01-02 09:00:00,Arapiraca,768,33.0
3,3,-36.664139,-9.768197,street_robbery,2022-01-02 23:00:00,Arapiraca,1850,47.0
4,4,-36.680604,-9.644294,street_robbery,2022-01-03 08:00:00,Arapiraca,2163,56.0
...,...,...,...,...,...,...,...,...
17366,17366,-36.662065,-9.764001,vehicle_robbery,2014-12-21 01:30:00,Arapiraca,1851,8497.0
17367,17367,-36.666260,-9.753766,street_robbery,2014-12-22 14:30:00,Arapiraca,1910,8534.0
17368,17368,-36.663400,-9.748675,street_robbery,2014-12-22 22:00:00,Arapiraca,1854,8542.0
17369,17369,-36.648386,-9.721165,vehicle_robbery,2014-12-22 23:30:00,Arapiraca,1633,8543.0


## 3. Tipos de crime e intervalo temporal

In [5]:
# Identifica os tipos de crime e o intervalo de anos nos dados

# Valor de GRID_SIZE calculado no notebook pre-processing-1
GRID_SIZE = 57

for ct in crimes_types:
    df_per_crime = df_hour[df_hour["ROUBO_GROUP"] == ct].copy()
    df_all_years_list = []
    for y in years:
        dfy = df_per_crime[df_per_crime["DATA_HORA"].dt.year == y].copy()

        # count crimes by (cell, DATA_HORA)
        counts = dfy.groupby(["cell", "hour_year"]).size()
        display(counts)

        grid = counts.unstack("hour_year") 


        beginning_of_y1 = datetime.datetime(y, 1, 1)
        beginning_of_y2 = datetime.datetime(y+1, 1, 1)
        num_hours_total = int((beginning_of_y2 - beginning_of_y1).total_seconds() // 3600)
        grid = grid.reindex(columns=range(num_hours_total))
        grid = grid.reindex(index=range(GRID_SIZE**2))
        df_year = grid.copy()
        df_year.columns = list(map(lambda x: str(x) + f"_{str(y)}", df_year.columns.tolist()))
        df_all_years_list.append(df_year)

  
    df_all = pd.concat(df_all_years_list,axis=1)
    df_all.T.to_csv(f"./output/arapiraca/arapiraca_{ct}_all_grid.csv")


cell  hour_year
817   1731.0       1
1339  7659.0       1
1513  7443.0       1
1563  2331.0       1
1567  2619.0       1
      2883.0       1
1568  5667.0       1
1571  3843.0       1
1573  1923.0       1
1620  4395.0       1
1680  2763.0       1
1682  2523.0       1
1736  2595.0       1
1737  2235.0       1
      2403.0       1
1738  7611.0       1
      8379.0       1
1740  243.0        2
1742  8547.0       1
1796  1707.0       1
      2403.0       1
      2643.0       1
      6675.0       1
      7155.0       1
      8307.0       1
1797  8067.0       1
1800  3171.0       1
      8595.0       1
1853  6891.0       1
      7275.0       1
      7563.0       1
1854  723.0        1
      1587.0       1
      2571.0       1
      6459.0       1
      8067.0       1
1857  315.0        1
1859  6843.0       1
1861  4011.0       1
1909  8379.0       1
1911  1803.0       1
1912  3003.0       1
1969  867.0        1
1970  51.0         1
2027  1563.0       1
2031  8019.0       1
2084  3987.0      

cell  hour_year
982   6833.0       1
1274  7629.0       1
1282  2102.0       1
      4104.0       1
1339  3985.0       1
                  ..
2089  6454.0       1
2142  8740.0       1
2145  8559.0       1
2201  2175.0       1
2545  3525.0       1
Length: 119, dtype: int64

cell  hour_year
1101  4480.0       1
1157  7213.0       1
1159  3668.0       1
1216  2919.0       1
1218  4101.0       1
                  ..
2089  7602.0       1
2142  2323.0       1
      2445.0       1
2145  436.0        1
2199  807.0        1
Length: 136, dtype: int64

cell  hour_year
1214  2725.0       1
1216  1753.0       1
      3017.0       1
      8683.0       1
1281  4555.0       1
                  ..
2089  1264.0       1
2140  8005.0       1
2142  7913.0       1
2146  1291.0       1
2264  3609.0       1
Length: 107, dtype: int64

cell  hour_year
239   8315.0       1
704   8315.0       1
705   2443.0       1
1098  8773.0       1
1215  8266.0       1
                  ..
2089  2130.0       1
      5685.0       1
2546  1215.0       1
2547  3279.0       1
2884  3499.0       1
Length: 153, dtype: int64

...

cell  hour_year
296   3088.0       1
410   8158.0       1
426   4002.0       1
518   1561.0       1
590   978.0        1
                  ..
2889  7362.0       1
2940  4077.0       1
2946  4511.0       1
3060  7887.0       1
3116  819.0        1
Length: 917, dtype: int64

cell  hour_year
461   3410.0       1
      6445.0       1
519   337.0        1
588   5635.0       1
      8445.0       1
                  ..
2831  8739.0       1
2883  1824.0       1
      2420.0       1
2888  4221.0       1
2951  5249.0       1
Length: 488, dtype: int64

cell  hour_year
0     3214.0       1
239   7049.0       1
254   860.0        1
363   3131.0       1
530   3946.0       1
                  ..
2717  2554.0       1
2883  8636.0       1
2884  8544.0       1
2941  885.0        1
3003  2520.0       1
Length: 307, dtype: int64

cell  hour_year
239   2738.0       1
295   6406.0       1
403   785.0        1
410   1390.0       1
461   118.0        1
                  ..
2831  6166.0       1
2883  6290.0       1
2884  865.0        1
      3838.0       1
      7534.0       1
Length: 347, dtype: int64

cell  hour_year
239   1190.0       1
645   3656.0       1
704   546.0        1
724   3474.0       1
768   33.0         1
                  ..
2720  5075.0       1
2775  3316.0       1
2831  4567.0       1
2884  4020.0       1
2951  2219.0       1
Length: 244, dtype: int64